In [ ]:
import pandas as pd
from datetime import date, timedelta, datetime
import xarray as xr
import numpy as np
import calendar
import os
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# ---- CONFIG ----
# Set these via environment variables, or edit directly for local runs.
# Data files are NOT included in this public repo (see README.md).
from pathlib import Path
import os

DATA_DIR = Path(os.environ.get("DELHI_DATA_DIR", "./data"))
MODEL_DIR = Path(os.environ.get("DELHI_MODEL_DIR", "./models"))
FIGURES_DIR = Path(os.environ.get("DELHI_FIGURES_DIR", "./figures"))
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
NTL_NAC_ZONE = pd.read_csv(str(DATA_DIR / "TTAV_Paper/cpcb_sations_lat_lon.csv"))#the attached file 

In [ ]:
NTL_NAC_ZONE=NTL_NAC_ZONE[NTL_NAC_ZONE['State']=='Delhi']

In [ ]:


def month_range(start_date, end_date):
    """Generates the first day of each month within a given date range."""
    current_date = start_date
    while current_date < end_date:
        yield current_date.replace(day=1)
        next_month = current_date.month + 1 if current_date.month < 12 else 1
        next_year = current_date.year if current_date.month < 12 else current_date.year + 1
        current_date = date(next_year, next_month, 1)

def get_filename(base_path, prefix, single_date):
    """Attempts to open a NetCDF file with different date formats."""
    base_path = Path(base_path)
    end_day = calendar.monthrange(single_date.year, single_date.month)[1]

    first_format = base_path / f"{prefix}_{single_date.year}-{single_date.month:02d}-01.nc"
    last_format = base_path / f"{prefix}_{single_date.year}-{single_date.month:02d}-{end_day}.nc"

    if first_format.exists():
        return xr.open_dataset(first_format)
    elif last_format.exists():
        return xr.open_dataset(last_format)
    else:
        print(f"Warning: No file found for {single_date.strftime('%Y-%m')} in {base_path}")
        return None  # Return None instead of crashing
def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days)):
        yield start_date + timedelta(n)
# Set date range
start_date = date(2019, 1, 1)
end_date = date(2024, 12,31)


final_data=pd.DataFrame()
for single_date in month_range(start_date, end_date):
    result_df = pd.DataFrame()#empty df to append
    alb = get_filename(str(DATA_DIR / "delhi/ERA/Forecast_albedo"), 'Forecast_albedo', single_date)
    if alb is None:
        continue  # Skip processing if file is missing

    cldfr_path = Path(fstr(DATA_DIR / "delhi/ERA/Fraction_of_cloud_cover/Fraction_of_cloud_cover_{single_date.year}-{single_date.month:02d}-01.nc"))
    if cldfr_path.exists():
        cldfr = xr.open_dataset(cldfr_path)
    else:
        print(f"Warning: Fraction_of_cloud_cover file missing for {single_date.strftime('%Y-%m')}")
        continue

    rh = get_filename(str(DATA_DIR / "delhi/ERA/Relative_humidity"), 'Relative_humidity', single_date)
    blh = get_filename(str(DATA_DIR / "/delhi/ERA/Boundary_layer_height"), 'boundary_layer_height', single_date)
    temp = get_filename(str(DATA_DIR / "delhi/ERA/Temperature"), 'Temperature', single_date)
    tcc = get_filename(str(DATA_DIR / "delhi/ERA/Total_cloud_cover"), 'Total_cloud_cover', single_date)
    tp = get_filename(str(DATA_DIR / "delhi/ERA/Total_precipitation"), 'Total_precipitation', single_date)
    sp = get_filename(str(DATA_DIR / "delhi/ERA/Surface_pressure"), 'Surface_pressure', single_date)
    u = get_filename(str(DATA_DIR / "delhi/ERA/U_component_of_wind"), 'U_component_of_wind', single_date)
    v = get_filename(str(DATA_DIR / "delhi/ERA/V_component_of_wind"), 'V_component_of_wind', single_date)
    TROPOMI = get_filename(str(DATA_DIR / "delhi/TROPOMI_NO2_varun"), 'TROPOMI_NO2', single_date)
    night_light = xr.open_dataset(fstr(DATA_DIR / "delhi/Night_light/Delhi_NTL_{single_date.year}_{single_date.month:02d}.nc"))
    emi=get_filename(str(DATA_DIR / "delhi/emission"), 'Total_emi_NOX', single_date) 
    ndvi=get_filename(str(DATA_DIR / "delhi/NDVI"), 'NDVI', single_date) 
    elevation = xr.open_dataset(str(DATA_DIR / "delhi/elevation/elevation_90.nc"))
    population = xr.open_dataset(str(DATA_DIR / "delhi/population/delhi_population.nc"))
    lulc = xr.open_dataset(fstr(DATA_DIR / "delhi/LULC/LULC_{single_date.year}.nc"))
    road = xr.open_dataset(str(DATA_DIR / "delhi/Road/delhi_road.nc"))
    

    lonvals = alb.lon.values
    latvals = alb.lat.values
    single_date = pd.to_datetime(single_date)
    # Create Dataset
    ds = xr.Dataset(
        data_vars=dict(
            albedo=(['lat', 'lon'],alb.fal[0].data),
            RelHum=(['lat', 'lon'],rh.r[0,0,:,:].data),
            Temp=(['lat', 'lon'], temp.t[0,0,:,:].data),
            CldFr=(['lat', 'lon'],cldfr.cc[0,0,:,:].data),
            Precipitation=(['lat', 'lon'],tp.tp[0].data),
            BLH=(['lat', 'lon'],blh.blh[0].data),
            SP=(['lat', 'lon'],sp.sp[0].data),
            UCW=(['lat', 'lon'],u.u[0,0,:,:].data),
            VCW=(['lat', 'lon'],v.v[0,0,:,:].data),
            TCC=(['lat', 'lon'],tcc.tcc[0].data),
            OMINO2=(['lat', 'lon'], TROPOMI.Band1.data),
            Elevation=(['lat', 'lon'], elevation['Band1'].data),
            Population=(['lat', 'lon'], population['Band1'].data),
            LULC=(['lat', 'lon'], lulc['LC_Prop1'][0].data),
            Night_Light=(['lat', 'lon'], night_light['Band1'].data),
            Emission=(['lat', 'lon'], emi['emissions'][0].data),
            NDVI=(['lat', 'lon'], ndvi['_250m_16_days_NDVI'][0,:,:].data),
            Road_Category=(['lat', 'lon'], road['Band1'].data)
        ),
        coords=dict(
            lat=latvals,
            lon=lonvals,
             time=[single_date],  # Add the single_date as a coordinate
        ),
        attrs=dict(description='Model data set'),
    )
    for idx in range(NTL_NAC_ZONE.shape[0]):
        geom = NTL_NAC_ZONE.iloc[idx,:]
        # Extract the data from ds using the geometry
        subset_ds = ds.sel(lat=geom.latitude, lon=geom.longitude, method='nearest')

        # Convert the subset_ds to a DataFrame
        subset_df = subset_ds.to_dataframe().reset_index()
        # Add a column for the feature's ID
        for ncolp in ['Station_Name', 'latitude','longitude','State','City']:
            subset_df[ncolp] = geom[ncolp]


        # Concatenate the results to the final DataFrame
        result_df = pd.concat([result_df, subset_df])
    final_data = pd.concat([final_data, result_df], ignore_index=True)
    print(single_date)



In [ ]:
cpcb=pd.read_csv(str(DATA_DIR / "TTAV_Paper/cpcb_hourly_data_2017_2024_update.csv"))

In [ ]:
cpcb['Timestamp']=pd.to_datetime(cpcb['Timestamp'],format='%Y-%m-%d %H:%M:%S')

In [ ]:
cpcb

In [ ]:

# Replace with the actual variable for NH₃

# Create a new DataFrame
df = cpcb[['Timestamp','PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO (µg/m³)','NO2 (µg/m³)','State']].copy()

In [ ]:
# Set Timestamp as index for easier resampling
df.set_index('Timestamp', inplace=True)

In [ ]:
df

In [ ]:
df.reset_index(inplace=True)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
import dash_bootstrap_components as dbc
from dash import dcc, html, Input, Output, dash_table
from itertools import cycle
from datetime import datetime as dt

# Load your data
# df = pd.read_csv('your_data.csv')

def process_hourly_to_daily(df):
    """Convert hourly data to daily statistics with additional metrics"""
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df.set_index('Timestamp', inplace=True)
    
    daily_data = df.groupby(['State']).resample('D').agg({
        'PM2.5 (µg/m³)': ['mean', 'std', 'max', 'min'],
        'PM10 (µg/m³)': ['mean', 'std', 'max'],
        'NO2 (µg/m³)': ['mean', 'std', 'max']
    })
    
    daily_data.columns = ['_'.join(col).strip() for col in daily_data.columns.values]
    daily_data.reset_index(inplace=True)
    
    # Calculate AQI categories
    def calculate_aqi_category(pm25):
        if pm25 <= 12: return 'Good'
        elif pm25 <= 35.4: return 'Moderate'
        elif pm25 <= 55.4: return 'Unhealthy for Sensitive Groups'
        elif pm25 <= 150.4: return 'Unhealthy'
        elif pm25 <= 250.4: return 'Very Unhealthy'
        else: return 'Hazardous'
    
    daily_data['AQI_Category'] = daily_data['PM2.5 (µg/m³)_mean'].apply(calculate_aqi_category)
    
    return daily_data.rename(columns={
        'PM2.5 (µg/m³)_mean': 'PM2.5_mean',
        'PM2.5 (µg/m³)_std': 'PM2.5_std',
        'PM2.5 (µg/m³)_max': 'PM2.5_max',
        'PM2.5 (µg/m³)_min': 'PM2.5_min',
        'PM10 (µg/m³)_mean': 'PM10_mean',
        'PM10 (µg/m³)_std': 'PM10_std',
        'PM10 (µg/m³)_max': 'PM10_max',
        'NO2 (µg/m³)_mean': 'NO2_mean',
        'NO2 (µg/m³)_std': 'NO2_std',
        'NO2 (µg/m³)_max': 'NO2_max'
    })

# Process data
daily_data = process_hourly_to_daily(df)

# Initialize Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Color schemes
state_colors = px.colors.qualitative.Plotly
aqi_colors = {
    'Good': '#00E400',
    'Moderate': '#FFFF00',
    'Unhealthy for Sensitive Groups': '#FF7E00',
    'Unhealthy': '#FF0000',
    'Very Unhealthy': '#8F3F97',
    'Hazardous': '#7E0023'
}

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1("Advanced Air Quality Dashboard", className="text-center my-4"), width=12)
    ]),
    
    # Control Panel
    dbc.Card([
        dbc.CardBody([
            dbc.Row([
                dbc.Col([
                    html.Label("Select States:", className="font-weight-bold"),
                    dcc.Dropdown(
                        id='state-selector',
                        options=[{'label': s, 'value': s} for s in daily_data['State'].unique()],
                        value=[daily_data['State'].unique()[0]],
                        multi=True,
                        clearable=False
                    )
                ], md=4),
                
                dbc.Col([
                    html.Label("Date Range:", className="font-weight-bold"),
                    dcc.DatePickerRange(
                        id='date-range',
                        min_date_allowed=daily_data['Timestamp'].min(),
                        max_date_allowed=daily_data['Timestamp'].max(),
                        start_date=daily_data['Timestamp'].min(),
                        end_date=daily_data['Timestamp'].max(),
                        display_format='YYYY-MM-DD'
                    )
                ], md=4),
                
                dbc.Col([
                    html.Label("Options:", className="font-weight-bold"),
                    dbc.Checklist(
                        id='std-toggle',
                        options=[{'label': ' Show Confidence Intervals', 'value': 'show'}],
                        value=[],
                        switch=True
                    ),
                    dbc.RadioItems(
                        id='plot-type',
                        options=[
                            {'label': ' Combined View', 'value': 'combined'},
                            {'label': ' Separate Views', 'value': 'separate'}
                        ],
                        value='combined',
                        inline=True
                    )
                ], md=4)
            ])
        ])
    ], className="mb-4"),
    
    # Key Metrics Cards
    dbc.Row([
        dbc.Col(dbc.Card(id='pm25-card', color="danger", inverse=True), md=4),
        dbc.Col(dbc.Card(id='pm10-card', color="warning", inverse=True), md=4),
        dbc.Col(dbc.Card(id='no2-card', color="info", inverse=True), md=4)
    ], className="mb-4"),
    
    # Main Visualization Area
    dbc.Row([
        dbc.Col(dcc.Graph(id='main-plot'), width=12, className="mb-4")
    ]),
    
    # Secondary Visualizations
    dbc.Row([
        dbc.Col(dcc.Graph(id='aqi-plot'), md=6),
        dbc.Col(dcc.Graph(id='correlation-plot'), md=6)
    ], className="mb-4"),
    
    # Data Table
    dbc.Row([
        dbc.Col([
            html.H4("Detailed Data", className="mb-3"),
            html.Div(id='summary-table')
        ], width=12)
    ]),
    
    # Footer
    dbc.Row([
        dbc.Col(html.P("Data last updated: " + dt.now().strftime("%Y-%m-%d %H:%M"), className="text-muted small"), width=12)
    ])
], fluid=True)

@app.callback(
    [Output('main-plot', 'figure'),
     Output('aqi-plot', 'figure'),
     Output('correlation-plot', 'figure'),
     Output('summary-table', 'children'),
     Output('pm25-card', 'children'),
     Output('pm10-card', 'children'),
     Output('no2-card', 'children')],
    [Input('state-selector', 'value'),
     Input('date-range', 'start_date'),
     Input('date-range', 'end_date'),
     Input('std-toggle', 'value'),
     Input('plot-type', 'value')]
)
def update_dashboard(selected_states, start_date, end_date, show_std, plot_type):
    if not selected_states:
        selected_states = daily_data['State'].unique()
    
    filtered = daily_data[
        (daily_data['State'].isin(selected_states)) & 
        (daily_data['Timestamp'] >= start_date) & 
        (daily_data['Timestamp'] <= end_date)
    ].copy()
    
    show_std = 'show' in show_std
    
    # Create summary statistics
    summary_stats = filtered.groupby('State').agg({
        'PM2.5_mean': ['mean', 'std', 'max'],
        'PM10_mean': ['mean', 'std', 'max'],
        'NO2_mean': ['mean', 'std', 'max'],
        'AQI_Category': lambda x: x.mode()[0] if not x.empty else None
    }).round(2).reset_index()
    
    summary_stats.columns = [' '.join(col).strip() for col in summary_stats.columns.values]
    
    # Create metric cards
    def create_metric_card(pollutant, color):
        avg_value = filtered[f'{pollutant}_mean'].mean()
        max_value = filtered[f'{pollutant}_max'].max()
        return dbc.CardBody([
            html.H4(pollutant, className="card-title"),
            html.H2(f"{avg_value:.1f} µg/m³", className="card-text"),
            html.P(f"Max: {max_value:.1f} µg/m³", className="card-text text-muted")
        ])
    
    pm25_card = create_metric_card('PM2.5', 'danger')
    pm10_card = create_metric_card('PM10', 'warning')
    no2_card = create_metric_card('NO2', 'info')
    
    # Assign colors to states
    state_color_map = {state: color for state, color in zip(selected_states, cycle(state_colors))}
    
    # Main Plot
    if plot_type == 'combined':
        fig = go.Figure()
        
        for state in selected_states:
            state_data = filtered[filtered['State'] == state]
            state_color = state_color_map[state]
            
            for pol, line_style in [('PM2.5', 'solid'), ('PM10', 'dot'), ('NO2', 'dash')]:
                fig.add_trace(go.Scatter(
                    x=state_data['Timestamp'],
                    y=state_data[f'{pol}_mean'],
                    mode='lines',
                    line=dict(color=state_color, dash=line_style),
                    name=f'{state} {pol}',
                    legendgroup=state,
                    showlegend=True,
                    hovertemplate=f"<b>{state} {pol}</b><br>Date: %{{x|%Y-%m-%d}}<br>Value: %{{y:.1f}} µg/m³<extra></extra>"
                ))
                
                if show_std:
                    fig.add_trace(go.Scatter(
                        x=state_data['Timestamp'],
                        y=state_data[f'{pol}_mean'] + state_data[f'{pol}_std'],
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False,
                        hoverinfo='skip',
                        legendgroup=state
                    ))
                    fig.add_trace(go.Scatter(
                        x=state_data['Timestamp'],
                        y=state_data[f'{pol}_mean'] - state_data[f'{pol}_std'],
                        mode='lines',
                        line=dict(width=0),
                        fillcolor=state_color.replace('rgb', 'rgba').replace(')', ',0.1)'),
                        fill='tonexty',
                        showlegend=False,
                        hoverinfo='skip',
                        legendgroup=state
                    ))
        
        fig.update_layout(
            title='Air Pollution Trends',
            xaxis_title='Date',
            yaxis_title='Concentration (µg/m³)',
            hovermode='x unified',
            height=500,
            plot_bgcolor='rgba(240,240,240,0.8)'
        )
    else:
        fig = make_subplots(rows=3, cols=1, subplot_titles=('PM2.5', 'PM10', 'NO2'))
        
        for i, pol in enumerate(['PM2.5', 'PM10', 'NO2'], 1):
            for state in selected_states:
                state_data = filtered[filtered['State'] == state]
                state_color = state_color_map[state]
                
                fig.add_trace(go.Scatter(
                    x=state_data['Timestamp'],
                    y=state_data[f'{pol}_mean'],
                    mode='lines',
                    line=dict(color=state_color),
                    name=f'{state} {pol}',
                    legendgroup=state,
                    showlegend=(i == 1),
                    hovertemplate=f"<b>{state} {pol}</b><br>Date: %{{x|%Y-%m-%d}}<br>Value: %{{y:.1f}} µg/m³<extra></extra>"
                ), row=i, col=1)
                
                if show_std:
                    fig.add_trace(go.Scatter(
                        x=state_data['Timestamp'],
                        y=state_data[f'{pol}_mean'] + state_data[f'{pol}_std'],
                        mode='lines',
                        line=dict(width=0),
                        showlegend=False,
                        hoverinfo='skip',
                        legendgroup=state
                    ), row=i, col=1)
                    fig.add_trace(go.Scatter(
                        x=state_data['Timestamp'],
                        y=state_data[f'{pol}_mean'] - state_data[f'{pol}_std'],
                        mode='lines',
                        line=dict(width=0),
                        fillcolor=state_color.replace('rgb', 'rgba').replace(')', ',0.1)'),
                        fill='tonexty',
                        showlegend=False,
                        hoverinfo='skip',
                        legendgroup=state
                    ), row=i, col=1)
        
        fig.update_layout(
            title='Air Pollution Metrics',
            height=800,
            showlegend=True,
            plot_bgcolor='rgba(240,240,240,0.8)'
        )
        fig.update_yaxes(title_text="Concentration (µg/m³)")
    
    # AQI Plot
    aqi_fig = px.bar(filtered, x='Timestamp', y='PM2.5_mean', color='AQI_Category',
                    color_discrete_map=aqi_colors,
                    title='Air Quality Index (Based on PM2.5)',
                    labels={'PM2.5_mean': 'PM2.5 (µg/m³)'},
                    hover_data={'State': True, 'PM2.5_mean': ':.1f'})
    
    aqi_fig.update_layout(
        plot_bgcolor='rgba(240,240,240,0.8)',
        legend_title_text='AQI Category'
    )
    
    # Correlation Plot - replaced scatter_matrix with pairwise scatter plots
    if len(selected_states) > 0 and len(filtered) > 0:
        corr_fig = make_subplots(rows=3, cols=1, 
                               subplot_titles=('PM2.5 vs PM10', 'PM2.5 vs NO2', 'PM10 vs NO2'))
        
        # PM2.5 vs PM10
        corr_fig.add_trace(
            go.Scatter(
                x=filtered['PM2.5_mean'],
                y=filtered['PM10_mean'],
                mode='markers',
                marker=dict(color=filtered['State'].map(state_color_map)),
                name='PM2.5 vs PM10',
                showlegend=False,
                hovertemplate="<b>%{text}</b><br>PM2.5: %{x:.1f}<br>PM10: %{y:.1f}<extra></extra>",
                text=filtered['State'] + '<br>' + filtered['Timestamp'].dt.strftime('%Y-%m-%d')
            ),
            row=1, col=1
        )
        
        # PM2.5 vs NO2
        corr_fig.add_trace(
            go.Scatter(
                x=filtered['PM2.5_mean'],
                y=filtered['NO2_mean'],
                mode='markers',
                marker=dict(color=filtered['State'].map(state_color_map)),
                name='PM2.5 vs NO2',
                showlegend=False,
                hovertemplate="<b>%{text}</b><br>PM2.5: %{x:.1f}<br>NO2: %{y:.1f}<extra></extra>",
                text=filtered['State'] + '<br>' + filtered['Timestamp'].dt.strftime('%Y-%m-%d')
            ),
            row=2, col=1
        )
        
        # PM10 vs NO2
        corr_fig.add_trace(
            go.Scatter(
                x=filtered['PM10_mean'],
                y=filtered['NO2_mean'],
                mode='markers',
                marker=dict(color=filtered['State'].map(state_color_map)),
                name='PM10 vs NO2',
                showlegend=False,
                hovertemplate="<b>%{text}</b><br>PM10: %{x:.1f}<br>NO2: %{y:.1f}<extra></extra>",
                text=filtered['State'] + '<br>' + filtered['Timestamp'].dt.strftime('%Y-%m-%d')
            ),
            row=3, col=1
        )
        
        corr_fig.update_layout(
            title_text='Pollutant Correlations',
            height=800,
            plot_bgcolor='rgba(240,240,240,0.8)',
            showlegend=False
        )
        
        # Add axis labels
        corr_fig.update_xaxes(title_text="PM2.5 (µg/m³)", row=1, col=1)
        corr_fig.update_yaxes(title_text="PM10 (µg/m³)", row=1, col=1)
        corr_fig.update_xaxes(title_text="PM2.5 (µg/m³)", row=2, col=1)
        corr_fig.update_yaxes(title_text="NO2 (µg/m³)", row=2, col=1)
        corr_fig.update_xaxes(title_text="PM10 (µg/m³)", row=3, col=1)
        corr_fig.update_yaxes(title_text="NO2 (µg/m³)", row=3, col=1)
    else:
        corr_fig = go.Figure()
        corr_fig.update_layout(
            title_text='No data available for correlation plot',
            plot_bgcolor='rgba(240,240,240,0.8)',
            height=300
        )
    
    # Summary Table
    table = dash_table.DataTable(
        data=summary_stats.to_dict('records'),
        columns=[{'name': col, 'id': col} for col in summary_stats.columns],
        style_table={'overflowX': 'auto'},
        style_cell={
            'textAlign': 'center',
            'padding': '5px',
            'minWidth': '100px'
        },
        style_header={
            'backgroundColor': 'rgb(230, 230, 230)',
            'fontWeight': 'bold'
        },
        style_data_conditional=[
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Good"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Good'],
                'color': 'black'
            },
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Moderate"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Moderate'],
                'color': 'black'
            },
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Unhealthy for Sensitive Groups"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Unhealthy for Sensitive Groups'],
                'color': 'white'
            },
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Unhealthy"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Unhealthy'],
                'color': 'white'
            },
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Very Unhealthy"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Very Unhealthy'],
                'color': 'white'
            },
            {
                'if': {
                    'filter_query': '{AQI_Category} = "Hazardous"',
                    'column_id': 'AQI_Category'
                },
                'backgroundColor': aqi_colors['Hazardous'],
                'color': 'white'
            }
        ],
        page_size=10,
        filter_action='native',
        sort_action='native'
    )
    
    return fig, aqi_fig, corr_fig, table, pm25_card, pm10_card, no2_card

if __name__ == '__main__':
    app.run_server(debug=True, port=8050)

In [ ]:
df.groupby('Timestamp')[['NO (µg/m³)','NO2 (µg/m³)']].mean().plot()

In [ ]:
daily_data = df.groupby(['State']).resample('D').mean()

In [ ]:
daily_data.to_csv(str(DATA_DIR / "cpcb_dailysatewise_data_2017_2024_update.csv"))

In [ ]:


# Step 1: Convert 'Time' to datetime if it isn't already
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Step 2: Set 'Time' as the index of the DataFrame
df.set_index('Timestamp', inplace=True)

# Step 3: Group by 'lat' and 'lon', resample by month, and compute the mean for 'NO2'
grouped = df.groupby(['latitude', 'longitude','Station_Name','State', 'City']).resample('M').mean()

# Step 4: Extract the first day of the month from the resampled index
grouped['First_day_of_month'] = grouped.index.get_level_values('Timestamp').to_period('M').start_time

# Step 5: Reset the index to turn 'lat', 'lon', and 'First_day_of_month' into columns
monthly_mean_df = grouped.reset_index()




In [ ]:

monthly_mean_df['Year'] = monthly_mean_df['Timestamp'].dt.year
monthly_mean_df['Month'] = monthly_mean_df['Timestamp'].dt.month
final_data['Year'] = final_data['time'].dt.year
final_data['Month'] = final_data['time'].dt.month

In [ ]:
final_data['time']=pd.to_datetime(final_data['time'])
final_data['NO2 (µg/m³)']=np.nan
final_data.set_index(['Year','Month','Station_Name','State', 'City'],inplace=True)
monthly_mean_df.set_index(['Year','Month','Station_Name','State', 'City'],inplace=True)
final_data.update(monthly_mean_df)

In [ ]:
final_data.reset_index(inplace=True)

In [ ]:
final_data.to_csv(str(DATA_DIR / "delhi/delhi_input_data_no2.csv"))

In [ ]:
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import r2_score, make_scorer
from scipy.stats import randint, uniform
from scikeras.wrappers import KerasRegressor
from tqdm import tqdm  # Progress bar
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

In [ ]:
data = pd.read_csv(str(DATA_DIR / "delhi/delhi_input_data_no2.csv"))



In [ ]:
# Data cleaning
Q1, Q3 = data['NO2 (µg/m³)'].quantile([0.25, 0.75])
IQR = Q3 - Q1
data = data[(data['NO2 (µg/m³)'] >= Q1 - 1.5 * IQR) & 
           (data['NO2 (µg/m³)'] <= Q3 + 1.5 * IQR)].dropna()

# Features and target
features = ['OMINO2', 'albedo', 'RelHum', 'Temp', 'CldFr', 'Precipitation', 'BLH', 'SP',
       'UCW', 'VCW', 'TCC', 'Elevation', 'Population', 'LULC',
       'Night_Light', 'Emission', 'NDVI', 'Road_Category', 'lat', 'lon','Year', 'Month']
target = 'NO2 (µg/m³)'

In [ ]:
data=data.dropna()

In [ ]:
len(data)

In [ ]:



X_train, X_test, y_train, y_test = train_test_split(data[features], data[target], test_size=0.1, random_state=42)

# Standardize the features
print("📌 Standardizing Data...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for LSTM input
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

# Define LSTM Model
def build_lstm_model(units=64, dropout_rate=0.2, optimizer='adam', learning_rate=0.001):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=(X_train_lstm.shape[1], 1)),
        Dropout(dropout_rate),
        LSTM(units // 2),
        Dropout(dropout_rate),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))
    return model

# Custom scoring function for R²
r2_scorer = make_scorer(r2_score)

# Wrap LSTM for hyperparameter tuning
lstm_regressor = KerasRegressor(model=build_lstm_model, verbose=0)

# Define hyperparameter grids
param_dist_lstm = {
    'model__units': [32, 64, 128],
    'model__dropout_rate': [0.1, 0.2, 0.3],
    'model__learning_rate': [0.0001, 0.001, 0.01],
    'batch_size': [8, 16, 32],
    'epochs': [10,20,30]
}

param_dist_rf = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}

param_dist_xgb = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4)
}

param_dist_lgbm = {
    'n_estimators': randint(50, 300),
    'max_depth': randint(3, 20),
    'learning_rate': uniform(0.01, 0.3),
    'num_leaves': randint(10, 50)
}

# Train models with tracking
models = {}

def run_model_search(model, params, name):
    print(f"🚀 Training {name} with Hyperparameter Tuning...")
    search = RandomizedSearchCV(
        model,
        param_distributions=params,
        n_iter=10,
        cv=3,
        verbose=10,
        n_jobs=-1,
        random_state=42,
        scoring=r2_scorer
    )
    search.fit(X_train_scaled, y_train)
    
    print(f"✅ {name} Best R²: {search.best_score_:.4f}")
    models[name] = search.best_estimator_
    return search

# Train models with CV and R² tracking
rf_search = run_model_search(RandomForestRegressor(random_state=42), param_dist_rf, "Random Forest")
xgb_search = run_model_search(xgb.XGBRegressor(objective='reg:squarederror', random_state=42), param_dist_xgb, "XGBoost")
lgbm_search = run_model_search(lgb.LGBMRegressor(random_state=42), param_dist_lgbm, "LightGBM")

# LSTM with RandomizedSearchCV
print("🚀 Training LSTM with Hyperparameter Tuning...")
lstm_search = RandomizedSearchCV(
    estimator=lstm_regressor,
    param_distributions=param_dist_lstm,
    n_iter=10,
    cv=3,
    verbose=10,
    n_jobs=-1,
    random_state=42,
    scoring=r2_scorer
)
lstm_search.fit(X_train_lstm, y_train)
models["lstm_model"] = lstm_search.best_estimator_


In [ ]:
# Train Linear Regression
print("🚀 Training Linear Regression...")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
models["linear_regression"] = lr_model
print("✅ Linear Regression Training Completed!")

In [ ]:
# Get the best model from RandomizedSearchCV
best_lstm_model = lstm_search.best_estimator_.model_ 
best_lstm_model.save(str(DATA_DIR / "model_stacking/Delhi/best_lstm_model.keras"))
print("✅ LSTM Model Saved Successfully in .keras format")

In [ ]:



from sklearn.metrics import r2_score, mean_absolute_error
# Get predictions from best models
print("📌 Generating Predictions...")
lstm_preds = models["lstm_model"].predict(X_test_lstm).flatten()
rf_preds = models["Random Forest"].predict(X_test_scaled)
xgb_preds = models["XGBoost"].predict(X_test_scaled)
lgbm_preds = models["LightGBM"].predict(X_test_scaled)
lr_preds = models["linear_regression"].predict(X_test_scaled)

# Create dataset for meta-model
meta_X_train = np.column_stack((lstm_preds, rf_preds, xgb_preds, lgbm_preds, lr_preds))

# Train Meta-Learner (LightGBM)
print("🚀 Training Meta-Learner (Stacking)...")
meta_model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
meta_model.fit(meta_X_train, y_test)
models["meta_model"] = meta_model
print("✅ Meta-Learner Training Completed!")



# Predictions & Evaluation
meta_preds = meta_model.predict(meta_X_train)
metrics = {
    'R² Score': r2_score(y_test, meta_preds),
    'Mean Absolute Error (MAE)': mean_absolute_error(y_test, meta_preds)
}
pd.DataFrame([metrics]).to_csv(str(DATA_DIR / "model_stacking/Delhi/model_metrics.csv"), index=False)

print(f"📊 Stacked Model R²: {metrics['R² Score']:.4f}")
print(f"📊 Stacked Model MAE: {metrics['Mean Absolute Error (MAE)']:.2f} µg/m³")
print("🎯 Training Complete! Models saved as 'best_models.pkl' 🎯")


In [ ]:
# # Save All Best Models
models["scaler"] = scaler
models["lstm_model_path"] = "best_lstm_model.h5"

joblib.dump(models, str(DATA_DIR / "model_stacking/Delhi/best_models.pkl"))  # Save all best models in one file

# # Save Best Hyperparameters
best_params = {
     "Random Forest": rf_search.best_params_,
     "XGBoost": xgb_search.best_params_,
     "LightGBM": lgbm_search.best_params_,
     "LSTM": lstm_search.best_params_
 }

pd.DataFrame([best_params]).to_csv(str(DATA_DIR / "model_stacking/Delhi/best_hyperparams.csv"), index=False)

In [ ]:
# Save NO₂ Measured vs. Predicted
no2_results = pd.DataFrame({
    "NO₂_Measured": y_test,
    "NO₂_Predicted": meta_preds
})
no2_results.to_csv(str(DATA_DIR / "model_stacking/Delhi/no2_measured_vs_predicted.csv"), index=False)

print("📊 NO₂ measured vs. predicted values saved as 'no2_measured_vs_predicted.csv'")


In [ ]:
no2_results=pd.read_csv(str(DATA_DIR / "model_stacking/Delhi/no2_measured_vs_predicted.csv"))

In [ ]:
no2_results.rename(columns={'NO2 (µg/m³)': 'NO₂_Measured', 'Predicted_NO2': 'NO₂_Predicted'}, inplace=True)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

# Set publication-quality parameters with BOLD FONT
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'font.weight': 'bold',  # <-- GLOBAL BOLD
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.dpi': 300,
    'figure.autolayout': True,
    'savefig.dpi': 300,
    'savefig.format': 'tiff',
    'savefig.bbox': 'tight'
})

# Calculate metrics
r2 = r2_score(no2_results['NO₂_Measured'], no2_results['NO₂_Predicted'])
rmse = np.sqrt(mean_squared_error(no2_results['NO₂_Measured'], no2_results['NO₂_Predicted']))
mae = mean_absolute_error(no2_results['NO₂_Measured'], no2_results['NO₂_Predicted'])
n = len(no2_results)

# Create figure
fig, ax = plt.subplots(figsize=(5, 5))

# Regression line (bold label)
sns.regplot(
    x='NO₂_Measured', 
    y='NO₂_Predicted', 
    data=no2_results,
    scatter=False,
    line_kws={
        'color': '#D62728',
        'linewidth': 1.5,
        'label': f'Linear fit (R² = {r2:.3f})'
    },
    ci=95,
    ax=ax
)

# Scatter points
ax.scatter(
    x=no2_results['NO₂_Measured'],
    y=no2_results['NO₂_Predicted'],
    s=40,
    alpha=0.7,
    edgecolor='k',
    linewidth=0.3,
    facecolor='#1F77B4',
    zorder=3
)

# Axis limits
buffer = 0.1 * (max_val - min_val)
min_val = min(no2_results[['NO₂_Measured', 'NO₂_Predicted']].min()) - buffer
max_val = max(no2_results[['NO₂_Measured', 'NO₂_Predicted']].max()) + buffer
ax.set_xlim(min_val, max_val)
ax.set_ylim(min_val, max_val)

# 1:1 line (bold in legend)
ax.plot([min_val, max_val], [min_val, max_val], '--', 
        c='#2CA02C', lw=1.5, alpha=0.7, label='1:1 line')

# Metrics annotation (BOLD TEXT)
textstr = '\n'.join((
    f'$N$ = {n}',
    f'$R^2$ = {r2:.2f}',
    f'RMSE = {rmse:.2f} µg m$^{{-3}}$',
    f'MAE = {mae:.2f} µg m$^{{-3}}$'
))

props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none')
ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=props, weight='bold')  # <-- EXPLICIT BOLD

# Axis labels (BOLD)
ax.set_xlabel('Measured NO$_2$ (µg m$^{-3}$)', weight='bold', labelpad=10)  # <-- BOLD
ax.set_ylabel('Predicted NO$_2$ (µg m$^{-3}$)', weight='bold', labelpad=10)  # <-- BOLD

# Grid and ticks (bold tick labels)
ax.grid(True, linestyle='--', alpha=0.4, zorder=0)
ax.tick_params(axis='both', which='both', direction='in', top=True, right=True, 
               labelsize=12, width=1.5)  # <-- BOLD TICKS via width

# Legend (bold text)
legend = ax.legend(loc='lower right', framealpha=0.9)
plt.setp(legend.get_texts(), weight='bold')  # <-- BOLD LEGEND TEXT

# Adjust layout
plt.tight_layout()

# Save figure (uncomment when ready)
plt.savefig(r'paper/NO2_Validation_Delhi_90m2.tiff', dpi=300)

plt.show()

## Data

In [ ]:
import os
import pickle
import joblib
import calendar
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from datetime import date
from tensorflow.keras.models import load_model
from pykrige.ok import OrdinaryKriging
import warnings
warnings.filterwarnings("ignore")


# Load all saved models
print("Loading saved models...")
# Load all models from the 'best_models.pkl' file
models = joblib.load(str(DATA_DIR / "model_stacking/Delhi/best_models.pkl"))


lstm_model = load_model(str(DATA_DIR / "model_stacking/Delhi/best_lstm_model.keras"))

print("✅ LSTM Model Loaded Successfully")

# Load scaler
scaler = models["scaler"]



In [ ]:


def month_range(start_date, end_date):
    """Generates the first day of each month within a given date range."""
    current_date = start_date
    while current_date < end_date:
        yield current_date.replace(day=1)
        next_month = current_date.month + 1 if current_date.month < 12 else 1
        next_year = current_date.year if current_date.month < 12 else current_date.year + 1
        current_date = date(next_year, next_month, 1)

def get_filename(base_path, prefix, single_date):
    """Attempts to open a NetCDF file with different date formats."""
    base_path = Path(base_path)
    end_day = calendar.monthrange(single_date.year, single_date.month)[1]

    first_format = base_path / f"{prefix}_{single_date.year}-{single_date.month:02d}-01.nc"
    last_format = base_path / f"{prefix}_{single_date.year}-{single_date.month:02d}-{end_day}.nc"

    if first_format.exists():
        return xr.open_dataset(first_format)
    elif last_format.exists():
        return xr.open_dataset(last_format)
    else:
        print(f"Warning: No file found for {single_date.strftime('%Y-%m')} in {base_path}")
        return None  # Return None instead of crashing
def daterange(start_date, end_date):
    for n in range(int((end_date - start_date).days)):
        yield start_date + timedelta(n)
# Set date range
start_date = date(2019, 1, 1)
end_date = date(2024,12,31)

# Initialize a list to store all predictions
all_predictions = []


final_data=pd.DataFrame()
for single_date in month_range(start_date, end_date):
    result_df = pd.DataFrame()#empty df to append
    alb = get_filename(str(DATA_DIR / "delhi/ERA/Forecast_albedo"), 'Forecast_albedo', single_date)
    if alb is None:
        continue  # Skip processing if file is missing

    cldfr_path = Path(fstr(DATA_DIR / "delhi/ERA/Fraction_of_cloud_cover/Fraction_of_cloud_cover_{single_date.year}-{single_date.month:02d}-01.nc"))
    if cldfr_path.exists():
        cldfr = xr.open_dataset(cldfr_path)
    else:
        print(f"Warning: Fraction_of_cloud_cover file missing for {single_date.strftime('%Y-%m')}")
        continue

    rh = get_filename(str(DATA_DIR / "delhi/ERA/Relative_humidity"), 'Relative_humidity', single_date)
    blh = get_filename(str(DATA_DIR / "/delhi/ERA/Boundary_layer_height"), 'boundary_layer_height', single_date)
    temp = get_filename(str(DATA_DIR / "delhi/ERA/Temperature"), 'Temperature', single_date)
    tcc = get_filename(str(DATA_DIR / "delhi/ERA/Total_cloud_cover"), 'Total_cloud_cover', single_date)
    tp = get_filename(str(DATA_DIR / "delhi/ERA/Total_precipitation"), 'Total_precipitation', single_date)
    sp = get_filename(str(DATA_DIR / "delhi/ERA/Surface_pressure"), 'Surface_pressure', single_date)
    u = get_filename(str(DATA_DIR / "delhi/ERA/U_component_of_wind"), 'U_component_of_wind', single_date)
    v = get_filename(str(DATA_DIR / "delhi/ERA/V_component_of_wind"), 'V_component_of_wind', single_date)
    TROPOMI = get_filename(str(DATA_DIR / "delhi/TROPOMI_NO2_varun"), 'TROPOMI_NO2', single_date)
    night_light = xr.open_dataset(fstr(DATA_DIR / "delhi/Night_light/Delhi_NTL_{single_date.year}_{single_date.month:02d}.nc"))
    emi=get_filename(str(DATA_DIR / "delhi/emission"), 'Total_emi_NOX', single_date) 
    ndvi=get_filename(str(DATA_DIR / "delhi/NDVI"), 'NDVI', single_date) 
    elevation = xr.open_dataset(str(DATA_DIR / "delhi/elevation/elevation_90.nc"))
    population = xr.open_dataset(str(DATA_DIR / "delhi/population/delhi_population.nc"))
    lulc = xr.open_dataset(fstr(DATA_DIR / "delhi/LULC/LULC_{single_date.year}.nc"))
    road = xr.open_dataset(str(DATA_DIR / "delhi/Road/delhi_road.nc"))
    

    lonvals = alb.lon.values
    latvals = alb.lat.values
    single_date = pd.to_datetime(single_date)
    # Create Dataset
    ds = xr.Dataset(
        data_vars=dict(
            albedo=(['lat', 'lon'],alb.fal[0].data),
            RelHum=(['lat', 'lon'],rh.r[0,0,:,:].data),
            Temp=(['lat', 'lon'], temp.t[0,0,:,:].data),
            CldFr=(['lat', 'lon'],cldfr.cc[0,0,:,:].data),
            Precipitation=(['lat', 'lon'],tp.tp[0].data),
            BLH=(['lat', 'lon'],blh.blh[0].data),
            SP=(['lat', 'lon'],sp.sp[0].data),
            UCW=(['lat', 'lon'],u.u[0,0,:,:].data),
            VCW=(['lat', 'lon'],v.v[0,0,:,:].data),
            TCC=(['lat', 'lon'],tcc.tcc[0].data),
            OMINO2=(['lat', 'lon'], TROPOMI.Band1.data),
            Elevation=(['lat', 'lon'], elevation['Band1'].data),
            Population=(['lat', 'lon'], population['Band1'].data),
            LULC=(['lat', 'lon'], lulc['LC_Prop1'][0].data),
            Night_Light=(['lat', 'lon'], night_light['Band1'].data),
            Emission=(['lat', 'lon'], emi['emissions'][0].data),
            NDVI=(['lat', 'lon'], ndvi['_250m_16_days_NDVI'][0,:,:].data),
            Road_Category=(['lat', 'lon'], road['Band1'].data)
        ),
        coords=dict(
            lat=latvals,
            lon=lonvals,
             time=[single_date],  # Add the single_date as a coordinate
        ),
        attrs=dict(description='Model data set'),
    )
    datatable = ds.to_dataframe()
    datalinear = datatable.reset_index()
    datafilled=datalinear.copy()
    datafilled['Elevation']=datafilled['Elevation'].interpolate(limit_direction='both')
    datafilled['Population']=datafilled['Population'].interpolate(limit_direction='both')
    datafilled['LULC']=datafilled['LULC'].interpolate(limit_direction='both')
    datafilled['OMINO2']=datafilled['OMINO2'].interpolate(limit_direction='both')
    datafilled['NDVI']=datafilled['NDVI'].interpolate(limit_direction='both')
    datafilled['Night_Light']=datafilled['Night_Light'].interpolate(limit_direction='both')
    datafilled['date']=single_date
    datafilled['date']=pd.to_datetime(datafilled['date'], format='%Y-%m-%d')
    datafilled['Year']=pd.to_datetime(datafilled['date']).dt.year
    datafilled['Month'] = pd.to_datetime(datafilled['date']).dt.month
    df=datafilled[['OMINO2', 'albedo', 'RelHum', 'Temp', 'CldFr', 'Precipitation', 'BLH', 'SP',
       'UCW', 'VCW', 'TCC', 'Elevation', 'Population', 'LULC',
       'Night_Light', 'Emission', 'NDVI', 'Road_Category', 'lat', 'lon','Year', 'Month']]
    print(single_date)
    
      # Select features in correct order
    X_pred = df.copy()
    # Scale features
    X_scaled = scaler.transform(X_pred)  # Apply the scaler to your features from df

    # Reshape data for LSTM input
    X_lstm = X_scaled.reshape((X_scaled.shape[0], X_scaled.shape[1], 1))

    lstm_preds = lstm_model.predict(X_lstm).flatten()
    rf_preds = models["Random Forest"].predict(X_scaled)
    xgb_preds = models["XGBoost"].predict(X_scaled)
    lgbm_preds = models["LightGBM"].predict(X_scaled)
    lr_preds = models["linear_regression"].predict(X_scaled)
    # Create a dataset for the meta-model
    meta_X = np.column_stack((lstm_preds, rf_preds, xgb_preds, lgbm_preds, lr_preds))

    # Get predictions from the meta-learner (stacked model)
    final_pred = models["meta_model"].predict(meta_X)

    
    # Create output dataset
    pred_df = pd.DataFrame({
        'lat': df['lat'],
        'lon': df['lon'],
        'NO2': final_pred
    })
    
    # Convert back to xarray with original grid
    full_grid = ds[['lat', 'lon']].to_dataframe().reset_index()[['lat', 'lon']]
    pred_df = full_grid.merge(pred_df, on=['lat', 'lon'], how='left')
    pred_ds = xr.Dataset.from_dataframe(pred_df.set_index(['lat', 'lon']))
    pred_ds = pred_ds.rename({'lat': 'latitude', 'lon': 'longitude'})
    pred_ds.coords['time'] = pd.to_datetime([single_date])
    pred_ds = pred_ds.set_coords(['latitude', 'longitude', 'time'])
    
    all_predictions.append(pred_ds)
    
# Concatenate all time steps into a single dataset
final_ds = xr.concat(all_predictions, dim="time")

# Save as a single NetCDF file
output_file = str(DATA_DIR / "delhi/delhi_NO2_Predictions_2019-2024_stacking_90m.nc")
final_ds.to_netcdf(output_file)



In [ ]:
final_ds['NO2'][0,:,]

In [ ]:
data=xr.open_dataset(str(DATA_DIR / "delhi/delhi_NO2_Predictions_2019-2024_stacking_90m_setgrid.nc"))

In [ ]:
mean_across_years = data.groupby('time.year').mean(dim='time')


annual_mean_overall = mean_across_years.mean(dim='year')

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import Normalize
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import cartopy.feature as cfeat
from cartopy.io.shapereader import Reader

# Load the shapefile
shape = Reader(str(DATA_DIR / "delhi/shape_file/Districts.shp"))
proj = ccrs.PlateCarree()
India = cfeat.ShapelyFeature(shape.geometries(), proj, edgecolor='k', facecolor='none')

# Define colormap and normalization
cmap = 'jet'  
vmin, vmax = 0, 60
norm = Normalize(vmin=vmin, vmax=vmax)

# Create figure with 2 rows × 3 columns
fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(18, 10), subplot_kw={'projection': ccrs.PlateCarree()})
title = [i for i in range(2019, 2025)]  # Years 2019 to 2024
plots = []

# Loop through subplots
for i, ax in enumerate(axs.flat[:6]):  
    ax.coastlines()
    ax.add_feature(India, linewidth=0.3, zorder=3)
    TRO = mean_across_years['NO2']
    
    # Plot NO2 data
    plot = TRO[i, :, :].plot.imshow(
        cmap=cmap, ax=ax, interpolation='bilinear', norm=norm, zorder=1, add_colorbar=False
    )
    
    ax.set_title(f'{title[i]}', fontsize=18, weight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)

    plots.append(plot)

# Set a global title
fig.suptitle(r'Annual Surface NO$_2$ Concentrations in Delhi (2019–2024)', 
             fontsize=20, weight='bold', y=1.05,x=0.1)


# Add shared colorbar with padding
cbar = fig.colorbar(plots[0], ax=axs, orientation='vertical', extend='max', aspect=40, pad=0.02)
cbar.set_label(r'NO$_2$ (µg/m$^3$)', fontsize=16)
cbar.ax.tick_params(labelsize=12)

# Improve spacing
plt.tight_layout(rect=[-0.1, 0, 0.78,1])  # Leave space for suptitle and colorbar
plt.show()


In [ ]:
data=data['NO2']

In [ ]:
# Select only December, January, February for each year across the 10-year period
djf_data = data.where( (data['time.month'] == 1) | (data['time.month'] == 2), drop=True)

# Calculate the mean across DJF for each year
djf_mean_across_years = djf_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
djf_mean_overall = djf_mean_across_years.mean(dim='year')

# Print the mean for DJF across the 10-year period
print(djf_mean_overall)

In [ ]:

MAM_data = data.where((data['time.month'] == 3) | (data['time.month'] == 4) | (data['time.month'] == 5), drop=True)

# Calculate the mean across DJF for each year
MAM_mean_across_years = MAM_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
MAM_mean_overall = MAM_mean_across_years.mean(dim='year')

In [ ]:
JAJ_data = data.where((data['time.month'] == 6) | (data['time.month'] == 7) | (data['time.month'] == 8) | (data['time.month'] == 9), drop=True)

# Calculate the mean across DJF for each year
JAJ_mean_across_years = JAJ_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
JAJ_mean_overall = JAJ_mean_across_years.mean(dim='year')

In [ ]:
SON_data = data.where( (data['time.month'] == 10) | (data['time.month'] ==11 | (data['time.month'] == 12) ), drop=True)

# Calculate the mean across DJF for each year
SON_mean_across_years = SON_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
SON_mean_overall = SON_mean_across_years.mean(dim='year')

In [ ]:
# Merge the datasets along a new dimension 'season'
combined_seasonal_means = xr.concat([djf_mean_overall,MAM_mean_overall,JAJ_mean_overall,SON_mean_overall], dim='season')

# Assign season names to the combined dataset
combined_seasonal_means['season'] = ['JF', 'MAM', 'JJAS', 'OND']

# Print the combined dataset
print(combined_seasonal_means)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import Normalize
import cartopy.feature as cfeat
from cartopy.io.shapereader import Reader
import matplotlib as mpl
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# Set bold serif fonts for publication-quality figure
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'font.weight': 'bold',
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'figure.constrained_layout.use': True
})

vmin, vmax = 0, 65
norm = Normalize(vmin=vmin, vmax=vmax)

# Create figure with adjusted dimensions
fig = plt.figure(figsize=(11, 4.5), dpi=300)

# Load Delhi districts shapefile
districts_shape = Reader(str(DATA_DIR / "delhi/shape_file/Districts.shp"))
delhi_extent = [76.8, 77.4, 28.4, 28.9]  # Delhi bounding box

# Create subplots
axs = []
for i in range(4):
    ax = fig.add_subplot(1, 4, i+1, projection=ccrs.PlateCarree())
    ax.set_extent(delhi_extent)
    axs.append(ax)

# Plotting parameters
cmap = 'RdYlBu_r'
seasons = ['Winter (DJF)', 'Pre-Monsoon (MAM)', 
           'Monsoon (JJAS)', 'Post-Monsoon (OND)']

for i, ax in enumerate(axs):
    img=combined_seasonal_means[i, :, :].plot(
        ax=ax,
        cmap=CMAP,
        add_colorbar=False,
        vmin=vmin,
        vmax=vmax,
        alpha=1.0
    )

    # Add district boundaries
    ax.add_geometries(districts_shape.geometries(), 
                     ccrs.PlateCarree(),
                     facecolor='none',
                     edgecolor='black',
                     linewidth=0.5)

    # Gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, 
                      color='gray', alpha=0.5, linestyle=':')
    gl.top_labels = gl.right_labels = False
    if i > 0:
        gl.left_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9, 'weight': 'bold'}
    gl.ylabel_style = {'size': 9, 'weight': 'bold'}

    ax.set_title(seasons[i], pad=8, fontweight='bold')

# Add horizontal colorbar
cbar_ax = fig.add_axes([0.15, 0.12, 0.7, 0.03])
cbar = fig.colorbar(img, cax=cbar_ax, orientation='horizontal',
                    label='NO₂ Surface Concentration (µg/m²)', extend='max')
cbar.ax.tick_params(labelsize=9, labelcolor='black')
cbar.set_label('NO₂ Surface Concentration (µg/m²)', fontsize=10, fontweight='bold')

# Main Title
fig.suptitle('Seasonal NO₂ Variations Across Delhi Districts (2019–2024)',
             y=1.05, fontsize=13, fontweight='bold')

# Adjust layout
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.25, top=0.85)
plt.tight_layout()
# Save figure
# plt.savefig('Delhi_Districts_NO2_Seasons.png', bbox_inches='tight', dpi=600)
plt.show()


In [ ]:


# Replace with your file paths
ds_india = xr.open_dataset(str(DATA_DIR / "model_stacking/NO2_Predictions_2019-2024_stacking_interpolated1KM.nc"))
ds_delhi = xr.open_dataset(str(DATA_DIR / "delhi/delhi_NO2_Predictions_2019-2024_stacking_90m_setgrid.nc"))

# Assume variable is named 'NO2', and time is monthly
no2_india = ds_india['NO2']
no2_delhi = ds_delhi['NO2']

# Compute annual climatology (mean over months)
no2_annual_india = no2_india.groupby('time.year').mean('time').mean('year')
no2_annual_delhi = no2_delhi.groupby('time.year').mean('time').mean('year')


In [ ]:
df=pd.read_csv(str(DATA_DIR / "cpcb_sations_lat_lon.csv"))

In [ ]:
df.columns

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.geoaxes
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# Create figure
fig = plt.figure(figsize=(14, 8))

# --- Main India Plot ---
ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot India NO2 data with consistent vmin/vmax
plot = no2_annual_india.plot(ax=ax, cmap='RdYlBu_r', 
                           add_colorbar=False, 
                           vmin=2, vmax=45)  # Using max of both datasets

# Add map features
ax.set_title('Annual NO₂ Climatology - India with Delhi Region (2019-2024)', pad=20, fontsize=14)
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_extent([68, 98, 6, 38])  # India bounding box

# Add India zone shapefile
india_shape = ShapelyFeature(Reader(str(DATA_DIR / "TTAV_Paper/Zone/zone.shp")).geometries(),
                           ccrs.PlateCarree(), facecolor='none', 
                           edgecolor='black', linewidth=1)
ax.add_feature(india_shape)

# --- Delhi Inset Plot (Larger) ---
# Create larger inset axes
axins = inset_axes(ax, width="60%", height="60%", loc='center left',bbox_to_anchor=(1.2, 0.2, 0.9, 0.9),
                    bbox_transform=ax.transAxes,
                  axes_class=cartopy.mpl.geoaxes.GeoAxes,
                  axes_kwargs=dict(map_projection=ccrs.PlateCarree()))

# Plot Delhi NO2 data with same color scaling
no2_annual_delhi.plot(ax=axins, cmap='RdYlBu_r',
                     add_colorbar=False,
                     vmin=2, vmax=45)
axins.set_title('Delhi Region', fontsize=12, pad=10)

# Add Delhi districts shapefile
districts_shape = ShapelyFeature(Reader(str(DATA_DIR / "delhi/shape_file/Districts.shp")).geometries(),
                               ccrs.PlateCarree(), facecolor='none',
                               edgecolor='black', linewidth=0.7)
axins.add_feature(districts_shape)

# Set Delhi extent (slightly wider area)
axins.set_extent([76.8, 77.48, 28.35, 28.92])  # More expanded view

# --- Visual Connection Between Plots ---
# Add rectangle showing Delhi location in main plot
delhi_extent = [76.8, 77.48, 28.35, 28.92]  # Match inset extent
rect = plt.Rectangle((delhi_extent[0], delhi_extent[2]),
                   delhi_extent[1]-delhi_extent[0],
                   delhi_extent[3]-delhi_extent[2],
                   fill=False, color='black', linewidth=1.5,
                   transform=ccrs.PlateCarree())
ax.add_patch(rect)

# Add connecting lines between main plot and inset
mark_inset(ax, axins, loc1=2, loc2=3, fc="none", ec="0.5", linestyle='--', linewidth=1.2)

# Add gridlines
ax.gridlines(draw_labels=True, linestyle='--', alpha=0.7)
axins.gridlines(draw_labels=True, linestyle='--', alpha=0.7)

# --- Single Colorbar ---
# Create axes for colorbar
cax = fig.add_axes([0.25, 0.05, 0.5, 0.03])  # [left, bottom, width, height]
cbar = fig.colorbar(plot, cax=cax, orientation='horizontal',extend='max')
cbar.set_label('NO₂ Concentration (µg/m³)', fontsize=12)
cbar.ax.tick_params(labelsize=10)

# Adjust layout and display
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)  # Make space for colorbar
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.geoaxes
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

delhi=df[df['State']=='Delhi'].copy()
lons_india = df.longitude.values
lats_india = df.latitude.values   

lons_delhi = delhi.longitude.values
lats_delhi = delhi.latitude.values
# Create figure
fig = plt.figure(figsize=(14, 8))

# --- Main India Plot ---
ax = plt.subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot India NO2 data with consistent vmin/vmax
plot = no2_annual_india.plot(ax=ax, cmap='RdYlBu_r', 
                           add_colorbar=False, 
                           vmin=2, vmax=45)

# Add scatter points for India
ax.scatter(lons_india, lats_india, 
           color='black', s=20,  # s=size of points
           edgecolor='white', linewidth=1,
           transform=ccrs.PlateCarree(),
           label='CPCB Ground Stations',
           zorder=10)  # Ensure points are on top

# Add map features
ax.set_title('Annual NO₂ Climatology - India with Delhi Region (2019-2024)', pad=20, fontsize=14)
ax.coastlines()
ax.add_feature(cfeature.BORDERS, linestyle=':')
ax.set_extent([68, 98, 6, 38])  # India bounding box

# Add India zone shapefile
india_shape = ShapelyFeature(Reader(str(DATA_DIR / "TTAV_Paper/Zone/zone.shp")).geometries(),
                           ccrs.PlateCarree(), facecolor='none', 
                           edgecolor='black', linewidth=1)
ax.add_feature(india_shape)

# Add legend for scatter points
ax.legend(loc='upper right', fontsize=10)

# --- Delhi Inset Plot ---
axins = inset_axes(ax, width="60%", height="60%", loc='center left',
                   bbox_to_anchor=(1.2, 0.2, 0.9, 0.9),
                   bbox_transform=ax.transAxes,
                   axes_class=cartopy.mpl.geoaxes.GeoAxes,
                   axes_kwargs=dict(map_projection=ccrs.PlateCarree()))

# Plot Delhi NO2 data
no2_annual_delhi.plot(ax=axins, cmap='RdYlBu_r',
                     add_colorbar=False,
                     vmin=2, vmax=45)

# Add scatter points for Delhi (make them slightly smaller)
axins.scatter(lons_delhi, lats_delhi, 
              color='black', s=25, 
              edgecolor='white', linewidth=0.8,
              transform=ccrs.PlateCarree(),
              zorder=10)

axins.set_title('Delhi Region', fontsize=12, pad=10)

# Add Delhi districts shapefile
districts_shape = ShapelyFeature(Reader(str(DATA_DIR / "delhi/shape_file/Districts.shp")).geometries(),
                               ccrs.PlateCarree(), facecolor='none',
                               edgecolor='black', linewidth=0.7)
axins.add_feature(districts_shape)

# Set Delhi extent
axins.set_extent([76.8, 77.48, 28.35, 28.92])

# --- Visual Connection Between Plots ---
delhi_extent = [76.8, 77.48, 28.35, 28.92]
rect = plt.Rectangle((delhi_extent[0], delhi_extent[2]),
                   delhi_extent[1]-delhi_extent[0],
                   delhi_extent[3]-delhi_extent[2],
                   fill=False, color='black', linewidth=1.5,
                   transform=ccrs.PlateCarree())
ax.add_patch(rect)

mark_inset(ax, axins, loc1=2, loc2=3, fc="none", ec="0.5", linestyle='--', linewidth=1.2)

# --- Gridlines and Labels ---
gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.7)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 10}
gl.ylabel_style = {'size': 10}

gl_inset = axins.gridlines(draw_labels=True, linestyle='--', alpha=0.7)
gl_inset.top_labels = False
gl_inset.right_labels = False
gl_inset.xlabel_style = {'size': 8}
gl_inset.ylabel_style = {'size': 8}

# Add axis labels
ax.set_xlabel('Longitude', fontsize=12, labelpad=10)
ax.set_ylabel('Latitude', fontsize=12, labelpad=10)
axins.set_xlabel('Longitude', fontsize=10, labelpad=5)
axins.set_ylabel('Latitude', fontsize=10, labelpad=5)

# --- Single Colorbar ---
cax = fig.add_axes([0.25, 0.05, 0.5, 0.03])
cbar = fig.colorbar(plot, cax=cax, orientation='horizontal', extend='max')
cbar.set_label('NO₂ Concentration (µg/m³)', fontsize=12)
cbar.ax.tick_params(labelsize=10)

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from matplotlib.patches import Rectangle
from matplotlib import ticker

# Set up figure with higher DPI for publication
fig = plt.figure(figsize=(14, 8), dpi=300)


# --- Main India Plot ---
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

# Plot with original RdYlBu_r colormap
plot = no2_annual_india.plot(ax=ax, cmap='RdYlBu_r', 
                           add_colorbar=False, 
                           vmin=2, vmax=45,
                           alpha=0.95)  # Slightly transparent

# Scatter points with improved visibility
scatter_india = ax.scatter(lons_india, lats_india, 
                         color='black', s=25,  
                         edgecolor='white', linewidth=0.8,
                         transform=ccrs.PlateCarree(),
                         label='All Monitoring Stations',
                         zorder=10)

# Highlight Delhi points in main plot
delhi_mask = df['State'] == 'Delhi'
ax.scatter(lons_india[delhi_mask], lats_india[delhi_mask],
           color='red', s=25, edgecolor='white', linewidth=0.8,
           transform=ccrs.PlateCarree(),
           label='Delhi Stations',
           zorder=11)

# Map features with improved styling
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.7)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linestyle=':', linewidth=0.7)
ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.3, edgecolor='gray')

# Add zone shapefile
india_shape = ShapelyFeature(Reader(str(DATA_DIR / "TTAV_Paper/Zone/zone.shp")).geometries(),
                           ccrs.PlateCarree(), facecolor='none', 
                           edgecolor='black', linewidth=1, linestyle='--')
ax.add_feature(india_shape)

# Title and labels
ax.set_title('Annual NO₂ Concentrations Across India (2019-2024)', 
             pad=15, fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=11, labelpad=10)
ax.set_ylabel('Latitude', fontsize=11, labelpad=10)

# --- Delhi Inset Plot (Right Side) ---
axins = inset_axes(ax, width="40%", height="40%", loc='center right',
                   bbox_to_anchor=(0.05,-0.25,0.95,0.95),  # Adjusted to right side
                   bbox_transform=ax.transAxes,
                   axes_class=cartopy.mpl.geoaxes.GeoAxes,
                   axes_kwargs=dict(map_projection=ccrs.PlateCarree()))

# Plot Delhi data with same colormap
no2_annual_delhi.plot(ax=axins, cmap='RdYlBu_r',
                     add_colorbar=False,
                     vmin=2, vmax=45,
                     alpha=0.95)

# Delhi scatter points with consistent styling
axins.scatter(lons_delhi, lats_delhi, 
             color='red', s=20, 
             edgecolor='white', linewidth=0.8,
             transform=ccrs.PlateCarree(),
             zorder=10,
             label='Monitoring Stations')

# Add district boundaries
districts_shape = ShapelyFeature(Reader(str(DATA_DIR / "delhi/shape_file/Districts.shp")).geometries(),
                               ccrs.PlateCarree(), facecolor='none',
                               edgecolor='black', linewidth=0.8)
axins.add_feature(districts_shape)

# Inset title and labels
axins.set_title('Delhi Metropolitan Region', fontsize=11, pad=10, fontweight='bold')
axins.set_xlabel('Longitude', fontsize=9, labelpad=5)
axins.set_ylabel('Latitude', fontsize=9, labelpad=5)

# --- Visual Connection Between Plots ---
delhi_extent = [76.8, 77.48, 28.35, 28.92]
rect = Rectangle((delhi_extent[0], delhi_extent[2]),
                delhi_extent[1]-delhi_extent[0],
                delhi_extent[3]-delhi_extent[2],
                fill=False, color='red', linewidth=1.5, linestyle='-',
                transform=ccrs.PlateCarree(), zorder=9)
ax.add_patch(rect)

# Connecting lines with better styling
mark_inset(ax, axins, loc1=1, loc2=2, fc="none", ec="0.3", 
           linestyle=':', linewidth=1.3)

# --- Gridlines and Labels ---
gl = ax.gridlines(draw_labels=True, linestyle=':', alpha=0.7)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 9}
gl.ylabel_style = {'size': 9}

gl_inset = axins.gridlines(draw_labels=True, linestyle=':', alpha=0.7)
gl_inset.top_labels = False
gl_inset.right_labels = False
gl_inset.xlabel_style = {'size': 8}
gl_inset.ylabel_style = {'size': 8}

# --- Combined Legend ---
# Create custom legend handles
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='CPCB Stations',
           markerfacecolor='black', markersize=8, markeredgecolor='white'),
    Line2D([0], [0], marker='o', color='w', label='CPCB Delhi Stations',
           markerfacecolor='red', markersize=8, markeredgecolor='white')
]

# Place legend in main plot
ax.legend(handles=legend_elements, loc='upper right', 
          fontsize=9, framealpha=1, 
          edgecolor='black', facecolor='white')

# --- Colorbar ---
cax = fig.add_axes([
    main_pos.x0 + main_pos.width + 0.02,  # Position right of main plot
    main_pos.y0,                          # Same bottom position
    0.02,                                 # Colorbar width
    main_pos.height                       # Match main plot height
])

cbar = fig.colorbar(plot, cax=cax, orientation='vertical', extend='max')
cbar.set_label('NO₂ Concentration (µg m$^{-3}$)', fontsize=11, labelpad=10)
cbar.ax.tick_params(labelsize=9)

# Add scale bars
def add_scale_bar(ax, length_km, loc=(0.1, 0.1)):
    """Add a scale bar to the map"""
    length_deg = length_km / 111
    ax.plot([loc[0], loc[0]+length_deg], [loc[1], loc[1]], 
            color='black', linewidth=2, transform=ccrs.PlateCarree())
    ax.text(loc[0]+length_deg/2, loc[1]-0.01, f'{length_km} km',
            ha='center', va='top', transform=ccrs.PlateCarree(),
            fontsize=8)

add_scale_bar(ax, 500, loc=(69, 7))  # Main map
add_scale_bar(axins, 20, loc=(76.79, 28.41))  # Inset

# Add north arrows
def add_north_arrow(ax, x, y, size):
    ax.annotate('N', xy=(x, y), xytext=(x, y-size),
                arrowprops=dict(facecolor='black', width=2, headwidth=8),
                ha='center', va='center', fontsize=10,
                transform=ccrs.PlateCarree())

add_north_arrow(ax, 95, 35, 2)
add_north_arrow(axins, 77.4, 28.9, 0.1)

# Adjust layout
# Adjust layout to prevent overlap with Delhi inset
plt.subplots_adjust(
    right=0.98,  # Make space for colorbar
    left=0.05    # Adjust left margin if needed
)

# Save as high-quality image
#plt.savefig('NO2_India_Delhi_RightInset.png', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
ds_india = xr.open_dataset(str(DATA_DIR / "model_stacking/NO2_Predictions_2019-2024_stacking_interpolated1KM.nc"))

In [ ]:
mean_across_years = ds_india.groupby('time.year').mean(dim='time')


annual_mean_overall = mean_across_years.mean(dim='year')

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib.colors import Normalize
import cartopy.crs as ccrs
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import cartopy.feature as cfeat
from cartopy.io.shapereader import Reader

# Set global font parameters for consistency
plt.rcParams.update({
    'font.size': 12,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 14,
    'axes.titlesize': 16,
    'figure.titlesize': 18
})

# Load the shapefile with error handling
try:
    shapefile_path = str(DATA_DIR / "TTAV_Paper/Zone/zone.shp")
    boundary_feature = cfeat.ShapelyFeature(
        Reader(shapefile_path).geometries(),
        ccrs.PlateCarree(),
        edgecolor='black',
        facecolor='none'
    )
except Exception as e:
    print(f"Error loading shapefile: {e}")
    boundary_feature = None

# Define visualization parameters
CMAP = 'RdYlBu_r'  # Reversed colormap for better perception
NO2_MIN, NO2_MAX =8,35  # Consistent value range across plots
YEAR_RANGE = range(2019, 2025)  # Years to display

# Create figure with proper dimensions for a paper
fig, axs = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(12, 8),  # Slightly smaller for paper columns
    subplot_kw={'projection': ccrs.PlateCarree()},
    dpi=300  # Higher resolution for publication
)

# Adjust figure spacing
fig.subplots_adjust(
    left=0.05,
    right=0.95,
    bottom=0.05,
    top=0.9,
    wspace=0.1,
    hspace=0.15
)

# Main plotting loop
plots = []
for idx, ax in enumerate(axs.flat):
    # Add base map features
    ax.add_feature(cfeat.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeat.BORDERS, linewidth=0.3, linestyle=':')
    if boundary_feature:
        ax.add_feature(boundary_feature, linewidth=0.5, zorder=3)
    
    # Plot NO2 data
    no2_data = mean_across_years['NO2'][idx, :, :]
    plot = no2_data.plot(
        ax=ax,
        cmap=CMAP,
        add_colorbar=False,
        vmin=NO2_MIN,
        vmax=NO2_MAX,
        alpha=1.0
    )
    
    # Add year title
    ax.set_title(f'{YEAR_RANGE[idx]}', fontweight='bold', pad=10)
    
    # Configure grid and ticks for better readability
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(),
        draw_labels=True,
        linewidth=0.5,
        color='gray',
        alpha=0.5,
        linestyle='--'
    )
    gl.top_labels = False
    gl.right_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 8}
    gl.ylabel_style = {'size': 8}
    
    plots.append(plot)

# Add main title with proper chemical notation
fig.suptitle(
    r'Annual Surface NO$_2$ Concentrations (2019–2024)',
    fontsize=16,
    fontweight='bold',
    y=1
)

# Add shared colorbar with proper labeling
cbar = fig.colorbar(
    plots[0],
    ax=axs,
    orientation='vertical',
    extend='both',  # Shows arrows for values beyond range
    shrink=0.8,
    aspect=30,
    pad=0.05
)
cbar.set_label(
    r'NO$_2$ Concentration (µg m$^{-3}$)',
    fontsize=12,
    labelpad=10
)
cbar.ax.tick_params(labelsize=10)

# Save figure in multiple formats for publication
# output_path = str(DATA_DIR / "TTAV_Paper/Figures/NO2_annual_concentrations")
# plt.savefig(f"{output_path}.png", dpi=300, bbox_inches='tight')

# Improve spacing
plt.tight_layout(rect=[-0.1, 0, 0.78,1])  
plt.show()

In [ ]:
data=ds_india['NO2']

In [ ]:
ds_india['NO2']

In [ ]:
# Select only December, January, February for each year across the 10-year period
djf_data = data.where( (data['time.month'] == 1) | (data['time.month'] == 2), drop=True)

# Calculate the mean across DJF for each year
djf_mean_across_years = djf_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
djf_mean_overall = djf_mean_across_years.mean(dim='year')

# Print the mean for DJF across the 10-year period
print(djf_mean_overall)

In [ ]:

MAM_data = data.where((data['time.month'] == 3) | (data['time.month'] == 4) | (data['time.month'] == 5), drop=True)

# Calculate the mean across DJF for each year
MAM_mean_across_years = MAM_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
MAM_mean_overall = MAM_mean_across_years.mean(dim='year')

In [ ]:
JAJ_data = data.where((data['time.month'] == 6) | (data['time.month'] == 7) | (data['time.month'] == 8) | (data['time.month'] == 9), drop=True)

# Calculate the mean across DJF for each year
JAJ_mean_across_years = JAJ_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
JAJ_mean_overall = JAJ_mean_across_years.mean(dim='year')

In [ ]:
SON_data = data.where( (data['time.month'] == 10) | (data['time.month'] ==11 | (data['time.month'] == 12) ), drop=True)

# Calculate the mean across DJF for each year
SON_mean_across_years = SON_data.groupby('time.year').mean(dim='time')

# Calculate the mean across all years for DJF
SON_mean_overall = SON_mean_across_years.mean(dim='year')

In [ ]:
# Merge the datasets along a new dimension 'season'
combined_seasonal_means = xr.concat([djf_mean_overall,MAM_mean_overall,JAJ_mean_overall,SON_mean_overall], dim='season')

# Assign season names to the combined dataset
combined_seasonal_means['season'] = ['JF', 'MAM', 'JJAS', 'OND']

# Print the combined dataset
print(combined_seasonal_means)

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import Normalize
import cartopy.feature as cfeat
from cartopy.io.shapereader import Reader
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# Set bold serif fonts for publication-quality figure
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'font.weight': 'bold',
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'figure.constrained_layout.use': True
})

# Data parameters
vmin, vmax = 2, 45
norm = Normalize(vmin=vmin, vmax=vmax)
cmap = 'RdYlBu_r'
seasons = ['Winter (DJF)', 'Pre-Monsoon (MAM)', 
           'Monsoon (JJAS)', 'Post-Monsoon (ON)']
delhi_extent = [68, 100, 5, 40]  # [min_lon, max_lon, min_lat, max_lat]

# Create figure with adjusted dimensions
fig = plt.figure(figsize=(11, 4.5), dpi=300)

# Load shapefile
districts_shape = Reader(str(DATA_DIR / "TTAV_Paper/Zone/zone.shp"))

# Create subplots
axs = []
for i in range(4):
    ax = fig.add_subplot(1, 4, i+1, projection=ccrs.PlateCarree())
    ax.set_extent(delhi_extent, crs=ccrs.PlateCarree())
    
    # Add basic map features
    ax.add_feature(cfeat.COASTLINE.with_scale('50m'), linewidth=0.5)
    ax.add_feature(cfeat.BORDERS.with_scale('50m'), linewidth=0.5)
    ax.add_feature(cfeat.STATES.with_scale('50m'), linewidth=0.3)
    
    axs.append(ax)

# Generate coordinate arrays for pcolormesh
lon = np.linspace(delhi_extent[0], delhi_extent[1], combined_seasonal_means.shape[2])
lat = np.linspace(delhi_extent[2], delhi_extent[3], combined_seasonal_means.shape[1])
lon_grid, lat_grid = np.meshgrid(lon, lat)

# Plot data
for i, ax in enumerate(axs):
    # Plot raster data using pcolormesh for proper projection handling
   
    
    img=combined_seasonal_means[i, :, :].plot(
        ax=ax,
        cmap=CMAP,
        add_colorbar=False,
        vmin=vmin,
        vmax=vmax,
        alpha=1.0
    )
    
    # Add district boundaries on top
    ax.add_geometries(districts_shape.geometries(), 
                     ccrs.PlateCarree(),
                     facecolor='none',
                     edgecolor='black',
                     linewidth=0.8,
                     zorder=10)  # High zorder ensures visibility

    # Configure gridlines
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, 
                     color='gray', alpha=0.5, linestyle=':')
    gl.top_labels = gl.right_labels = False
    if i > 0:
        gl.left_labels = False
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {'size': 9, 'weight': 'bold'}
    gl.ylabel_style = {'size': 9, 'weight': 'bold'}

    ax.set_title(seasons[i], pad=8, fontweight='bold')

# Add colorbar
cbar_ax = fig.add_axes([0.15, 0.08, 0.75, 0.04])
cbar = fig.colorbar(img, cax=cbar_ax, orientation='horizontal',
                   extend='max')
cbar.set_label('NO₂ Surface Concentration (µg/m³)', 
              fontsize=10, fontweight='bold')
cbar.ax.tick_params(labelsize=9)

# Main title
fig.suptitle('Seasonal NO₂ Variations (2019-2024)',
            y=1.05, fontsize=13, fontweight='bold')

# Adjust layout
plt.subplots_adjust(left=0.05, right=0.95, bottom=0.25, top=0.85)


# Save figure
# plt.savefig('Seasonal_NO2_Variations.png', 
#            bbox_inches='tight', 
#            dpi=600, 
#            facecolor='white')
plt.show()